In [ ]:
# All imports and configuration. Install requirements outside this notebook.
import copy, csv, datetime as dt, gc, hashlib, json, math, os, platform, random, shutil, subprocess, sys, time, uuid, zipfile
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from PIL import Image
from scipy import ndimage as ndi
from numba import njit
from pycocotools import mask as mask_utils
from skimage.metrics import structural_similarity
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
ID='17-spatial-attack-training'
IN_COLAB=Path('/content').is_dir() and sys.platform=='linux'
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/ID/'experiment.ipynb').is_file()),None)
assert ROOT is not None, 'Run the notebook from the repository root or its src directory'
EXP=ROOT/'src'/ID
PROFILE=os.environ.get('UNIFIED_PROFILE','smoke')
assert PROFILE in ('prepare','smoke','benchmark','train','freeze')
if not IN_COLAB: assert Path(sys.prefix).resolve()==(EXP/'.venv').resolve(), 'Use own experiment .venv'
RUN_ID=os.environ.get('UNIFIED_RUN_ID', 'unified17-seed17-v1' if PROFILE in ('train','freeze') else PROFILE+'-'+uuid.uuid4().hex[:8])
assert Path(RUN_ID).name==RUN_ID
RUN=EXP/'runs'/RUN_ID
DATA=ROOT/'data/unified17-coco'
ARCHIVE=ROOT/'data/unified17-coco.zip'
DRIVE_ARCHIVE=Path('/content/drive/MyDrive/Data/unified17-coco.zip')
SIZE,BITS,CHANNELS,AMPLITUDE=128,16,64,.03
IMAGE_SIZE=SIZE; MAX_MASK_PIXELS=2458
SEED=17; DATA_SEED=171201; MASK_SEED=171202; VAL_SEED=171203; TEST_SEED=181203
AREAS=[819,1638,2458]; SIGMAS=[.02,.05,.1]
BATCH=int(os.environ.get('UNIFIED_BATCH','16')); EVAL_BATCH=64
EPOCHS=int(os.environ.get('UNIFIED_EPOCHS','20')); PARENT_EPOCHS=40
LR,SIGMA,CLIP=.001,.05,5.
TRAIN_DEADLINE=time.time()+float(os.environ.get('UNIFIED_TRAIN_HOURS','8'))*3600
OVERALL_DEADLINE=time.time()+float(os.environ.get('UNIFIED_TOTAL_HOURS','12'))*3600
ARM_CONFIG={f'{family}-obj{int(p*100):03d}':dict(family=family,p=p,random_mask=False) for family in ('gaussian','erase','mix') for p in (0.,.1,.5,.9,1.)}
ARM_CONFIG.update({'clean':dict(family='clean',p=.5,random_mask=False),'global':dict(family='global',p=.5,random_mask=False)})
ARM_CONFIG.update({f'random-{f}':dict(family=f,p=.5,random_mask=True) for f in ('gaussian','erase','mix')})
ARMS=list(ARM_CONFIG)
assert len(ARMS)==20
CUDA=PROFILE!='prepare' and torch.cuda.is_available()
DEVICE=torch.device('cuda' if CUDA else 'cpu'); AMP=CUDA
if PROFILE in ('train','benchmark'): assert IN_COLAB and CUDA and any(x in torch.cuda.get_device_name() for x in ('A100','T4'))
torch.set_num_threads(4); torch.backends.cudnn.benchmark=CUDA; torch.backends.cudnn.deterministic=False
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if CUDA: torch.cuda.manual_seed_all(SEED)
CONFIG=dict(experiment=ID,seed=SEED,size=SIZE,bits=BITS,channels=CHANNELS,amplitude=AMPLITUDE,arms=ARM_CONFIG,batch=BATCH,epochs=EPOCHS,parent_epochs=PARENT_EPOCHS,lr=LR,sigma=SIGMA,clip=CLIP,areas=AREAS,selection='min max(object/background gaussian/erase BER), then mean BCE; PSNR>=30',train_deadline=TRAIN_DEADLINE,overall_deadline=OVERALL_DEADLINE,precision='FP16 AMP' if AMP else 'FP32')


Описание: [обучение пространственных режимов](../../README.md). Подготовка не делает предсказаний; обучение не загружает test.


In [ ]:
def sha(path):
    with Path(path).open('rb') as f: return hashlib.file_digest(f,'sha256').hexdigest()
def write(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.partial'); temp.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n'); os.replace(temp,path)
def save(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.partial');torch.save(value,tmp);os.replace(tmp,path)
def state(model):return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
def digest(weights):
    h=hashlib.sha256()
    for k,v in sorted(weights.items()):h.update(k.encode());h.update(v.detach().cpu().numpy().tobytes())
    return h.hexdigest()
def csvwrite(path,rows):
    with Path(path).open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def sync():
    if CUDA:torch.cuda.synchronize()
def psnr(mse):return -10*math.log10(max(float(mse),1e-12))
def stamp(status,**kwargs):write(RUN/'progress.json',dict(status=status,time=dt.datetime.now(dt.timezone.utc).isoformat(),**kwargs))
def deadline():
    if time.time()>TRAIN_DEADLINE:raise TimeoutError('Training deadline reached; checkpoints preserved')
RUN.mkdir(parents=True,exist_ok=True)
SOURCE_SHA=sha(EXP/'experiment.ipynb')
if (RUN/'source.ipynb').exists():assert sha(RUN/'source.ipynb')==SOURCE_SHA,'Resume source changed'
else:shutil.copyfile(EXP/'experiment.ipynb',RUN/'source.ipynb')
write(RUN/'config.json',CONFIG)
(RUN/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
write(RUN/'environment.json',dict(python=sys.version,torch=str(torch.__version__),gpu=torch.cuda.get_device_name() if CUDA else None,platform=platform.platform(),source_sha=SOURCE_SHA))


In [ ]:
def largest_component(region):
    labels,n=ndi.label(region,structure=np.array([[0,1,0],[1,1,1],[0,1,0]]))
    if not n:return np.zeros_like(region,dtype=bool),0
    counts=np.bincount(labels.ravel());counts[0]=0;chosen=int(counts.argmax())
    return labels==chosen,int(counts[chosen])

@njit(nogil=True)
def bfs_mask(allowed,origin,k,width):
    total=allowed.size;queue=np.empty(total,np.int32);seen=np.zeros(total,np.uint8);mask=np.zeros(total,np.uint8)
    if k==0:return mask
    queue[0]=origin;seen[origin]=1;tail=1;head=0
    while head<k and head<tail:
        index=queue[head];head+=1;mask[index]=1
        y=index//width;x=index%width
        for direction in range(4):
            nxt=-1
            if direction==0 and y>0:nxt=index-width
            elif direction==1 and x>0:nxt=index-1
            elif direction==2 and index+width<total:nxt=index+width
            elif direction==3 and x+1<width:nxt=index+1
            if nxt>=0 and allowed[nxt] and not seen[nxt]:
                seen[nxt]=1;queue[tail]=nxt;tail+=1
    if head!=k:raise ValueError('Component smaller than requested area')
    return mask

def exact_mask(component,seed,k):
    if not k:return np.zeros_like(component,dtype=bool)
    coords=np.flatnonzero(component)
    origin=int(np.random.default_rng(seed).choice(coords))
    result=bfs_mask(component.ravel().astype(np.uint8),origin,k,component.shape[1]).reshape(component.shape).astype(bool)
    assert int(result.sum())==k and not np.any(result&~component)
    return result

def semantic_components(info,annotations):
    foreground=np.zeros((IMAGE_SIZE,IMAGE_SIZE),bool);all_things=foreground.copy();crowd=foreground.copy()
    for ann in annotations:
        seg=ann.get('segmentation')
        if not seg:continue
        h,w=int(info['height']),int(info['width'])
        if isinstance(seg,list):rle=mask_utils.merge(mask_utils.frPyObjects(seg,h,w))
        elif isinstance(seg['counts'],list):rle=mask_utils.frPyObjects(seg,h,w)
        else:rle=seg
        native=mask_utils.decode(rle)
        if native.ndim==3:native=native.any(axis=2).astype(np.uint8)
        region=np.asarray(Image.fromarray(native).resize((IMAGE_SIZE,IMAGE_SIZE),Image.Resampling.NEAREST),dtype=bool)
        all_things|=region
        if ann.get('iscrowd',0):crowd|=region
        else:foreground|=region
    obj,oa=largest_component(foreground&~crowd);bg,ba=largest_component(~all_things)
    return np.stack([obj,bg]),min(MAX_MASK_PIXELS,oa,ba),[oa,ba]


In [ ]:
def recursive_ids(value):
    out=set()
    if isinstance(value,dict):
        for k,v in value.items():
            if k=='image_ids' and isinstance(v,list):out.update(int(i) for i in v)
            else:out.update(recursive_ids(v))
    elif isinstance(value,list):
        for v in value:out.update(recursive_ids(v))
    elif isinstance(value,str) and value.endswith('.jpg') and Path(value).stem.isdigit():out.add(int(Path(value).stem))
    return out

def shifted_mask(mask,rng):
    yy,xx=np.where(mask); y=yy-yy.min();x=xx-xx.min()
    dy=int(rng.integers(SIZE-int(y.max())));dx=int(rng.integers(SIZE-int(x.max())))
    out=np.zeros_like(mask);out[y+dy,x+dx]=1
    assert out.sum()==mask.sum() and largest_component(out)[1]==mask.sum()
    return out

def prepare():
    if (DATA/'identity.json').exists():
        pin=json.loads((DATA/'identity.json').read_text())
        for f,h in pin['files'].items():assert sha(DATA/f)==h
        return pin
    DATA.mkdir(parents=True,exist_ok=True)
    hist=set(); sources=[]
    for p in [ROOT/'data/problem1-coco-5000-500-500/manifest.json',ROOT/'data/problem1-coco-5000-500-500/historical-exclusions.json',ROOT/'data/coco2017-hidden-1800-200/manifest.json']+list((ROOT/'data/cache/coco2017/manifests').glob('*.json')):
        if p.exists(): hist.update(recursive_ids(json.loads(p.read_text())));sources.append(dict(path=str(p.relative_to(ROOT)),sha256=sha(p)))
    npz=ROOT/'data/problem1-coco-5000-500-500/15-weak-bit-reweighting/test.npz'
    if npz.exists():
        with np.load(npz) as bank:
            for k in ('names','ids','image_ids'):
                if k in bank:hist.update(int(Path(str(x)).stem) for x in bank[k])
        sources.append(dict(path=str(npz.relative_to(ROOT)),sha256=sha(npz)))
    write(DATA/'historical-exclusions.json',dict(image_ids=sorted(hist),sources=sources))
    native=ROOT/'data/raw/coco2017'
    # Historical RGB hash audit; IDs already excluded before decoding candidates.
    past_hash=set()
    for i in sorted(hist):
        name=f'{i:012d}.jpg';p=native/'train2017'/name
        if not p.exists():p=native/'val2017'/name
        if p.exists():
            with Image.open(p) as im:a=np.array(im.convert('RGB').resize((SIZE,SIZE),Image.Resampling.BILINEAR))
            past_hash.add(hashlib.sha256(a.tobytes()).hexdigest())
    chosen={}; seen=set(past_hash); annotation_hashes={}; refused=defaultdict(int)
    bfs_mask(np.ones(4,np.uint8),0,2,2)
    for official, targets in [('train2017',[('train',5000),('val',500)]),('val2017',[('test',500)])]:
        ap=native/'annotations'/f'instances_{official}.json'
        annotation_hashes[official]=sha(ap)
        ann=json.loads(ap.read_text()); infos={int(i['id']):i for i in ann['images']}; byid=defaultdict(list)
        for a in ann['annotations']:byid[int(a['image_id'])].append(a)
        candidates=np.array(sorted(set(infos)-hist));np.random.default_rng(DATA_SEED+(official=='val2017')).shuffle(candidates)
        def candidate(i):
            i=int(i);components,k,aa=semantic_components(infos[i],byid[i])
            if k!=MAX_MASK_PIXELS:return None,'area'
            p=native/official/infos[i]['file_name']
            with Image.open(p) as im:rgb=np.array(im.convert('RGB').resize((SIZE,SIZE),Image.Resampling.BILINEAR))
            return (i,rgb,components,aa,hashlib.sha256(rgb.tobytes()).hexdigest()),None
        ti=0;split,needed=targets[ti];chosen[split]=[]
        # bounded chunks avoid enqueuing every COCO image.
        with ThreadPoolExecutor(max_workers=8) as pool:
            for offset in range(0,len(candidates),128):
                for item,reason in pool.map(candidate,candidates[offset:offset+128]):
                    if reason:refused[official+'_'+reason]+=1;continue
                    if item[-1] in seen:refused[official+'_pixelduplicate']+=1;continue
                    seen.add(item[-1]);chosen[split].append(item)
                    if len(chosen[split])==needed:
                        print('selected',split,needed,flush=True)
                        ti+=1
                        if ti==len(targets):break
                        split,needed=targets[ti];chosen[split]=[]
                if ti==len(targets):break
        assert ti==len(targets), 'Insufficient eligible images'
        del ann,byid,infos;gc.collect()
    manifest=dict(size=SIZE,bits=BITS,mask_pixels=MAX_MASK_PIXELS,selection_seed=DATA_SEED,annotation_hashes=annotation_hashes,historical_count=len(hist),historical_rgb_hashes=len(past_hash),refused=dict(refused),splits={})
    for split,items in chosen.items():
        n=len(items);variants=EPOCHS if split=='train' else len(AREAS)
        masks=np.lib.format.open_memmap(DATA/f'{split}_masks.npy',mode='w+',dtype=np.uint8,shape=(variants,n,3,SIZE*SIZE//8))
        rgb=np.stack([x[1] for x in items]);np.save(DATA/f'{split}.npy',rgb)
        coverage=np.zeros((variants,n),np.float32)
        def make(pair):
            j,(i,_,components,aa,h)=pair;packed=np.zeros((variants,3,SIZE*SIZE//8),np.uint8);frac=[]
            for v in range(variants):
                k=MAX_MASK_PIXELS if split=='train' else AREAS[v]
                # Fixed origin across eval areas gives nested masks.
                key=MASK_SEED+i*131+(v*1000003 if split=='train' else 0)
                mm=[exact_mask(components[r],key+r*71,k) for r in (0,1)]
                for r,m in enumerate(mm):assert largest_component(m)[1]==k and not np.any(m&~components[r])
                assert not np.any(mm[0]&mm[1])
                rnd=shifted_mask(mm[(i+v)%2],np.random.default_rng(key+901))
                frac.append(float((rnd&components[0]).sum()/k))
                packed[v]=np.stack([np.packbits(m.ravel()) for m in mm+[rnd]])
            return j,packed,frac
        with ThreadPoolExecutor(max_workers=8) as pool:
            for j,packed,frac in pool.map(make,enumerate(items)):
                masks[:,j]=packed;coverage[:,j]=frac
                if j%1000==0:print('masks',split,j,flush=True)
        masks.flush();del masks
        np.save(DATA/f'{split}_random_overlap.npy',coverage)
        manifest['splits'][split]=dict(count=n,ids=[x[0] for x in items],rgb_sha256=[x[-1] for x in items],component_areas=[x[3] for x in items])
        if split!='train':np.save(DATA/f'{split}_bits.npy',np.random.default_rng(VAL_SEED if split=='val' else TEST_SEED).integers(0,2,(n,BITS),dtype=np.uint8))
    sets=[set(x['ids']) for x in manifest['splits'].values()]
    assert sum(map(len,sets))==len(set.union(*sets))==6000 and not set.union(*sets)&hist
    write(DATA/'manifest.json',manifest)
    pin=dict(files={p.name:sha(p) for p in sorted(DATA.iterdir()) if p.is_file() and p.name!='identity.json'},counts={s:len(v) for s,v in chosen.items()})
    write(DATA/'identity.json',pin)
    with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for p in sorted(DATA.iterdir()):z.write(p,DATA.name+'/'+p.name)
    Path(str(ARCHIVE)+'.sha256').write_text(sha(ARCHIVE)+'\n')
    write(RUN/'preparation.json',dict(identity=pin,archive_sha256=sha(ARCHIVE),archive_bytes=ARCHIVE.stat().st_size))
    print('dataset complete',pin['counts'],flush=True)
    return pin

def load_data():
    if not DATA.exists():
        if not ARCHIVE.exists():
            ARCHIVE.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(DRIVE_ARCHIVE,ARCHIVE);shutil.copyfile(str(DRIVE_ARCHIVE)+'.sha256',str(ARCHIVE)+'.sha256')
        assert sha(ARCHIVE)==Path(str(ARCHIVE)+'.sha256').read_text().split()[0]
        with zipfile.ZipFile(ARCHIVE) as z:
            assert all(n.startswith(DATA.name+'/') and '..' not in Path(n).parts for n in z.namelist());z.extractall(DATA.parent)
    pin=json.loads((DATA/'identity.json').read_text())
    # Deliberately do not open test arrays during training.
    for name,h in pin['files'].items():
        if not name.startswith('test'):assert sha(DATA/name)==h
    arrays={s:torch.from_numpy(np.load(DATA/f'{s}.npy')).permute(0,3,1,2).contiguous() for s in ('train','val')}
    if CUDA:arrays={s:a.pin_memory() for s,a in arrays.items()}
    banks={s:np.load(DATA/f'{s}_masks.npy',mmap_mode='r') for s in ('train','val')}
    words=np.load(DATA/'val_bits.npy');manifest=json.loads((DATA/'manifest.json').read_text())
    return arrays,banks,words,manifest

def mask_batch(banks,split,ids,variant,regions):
    packed=banks[split][variant,np.asarray(ids),np.asarray(regions)]
    return torch.from_numpy(np.unpackbits(packed,axis=-1).reshape(-1,1,SIZE,SIZE).copy()).to(DEVICE,dtype=torch.float32)
def image_batch(arrays,split,ids):return arrays[split][ids].to(DEVICE,dtype=torch.float32,non_blocking=CUDA,memory_format=torch.channels_last)/255.


In [ ]:
def normalization(ch):
 return nn.BatchNorm2d(ch)

class Block(nn.Sequential):
 def __init__(self,a,b):super().__init__(nn.Conv2d(a,b,3,padding=1,bias=False),normalization(b),nn.ReLU())

class CNN(nn.Module):
 def __init__(self,spatial=False):
  super().__init__();self.spatial=spatial;self.message_map=nn.Linear(BITS,BITS*16*16) if spatial else nn.Identity();self.features=nn.Sequential(*[Block(3 if i==0 else CHANNELS,CHANNELS) for i in range(4)])
  self.fuse=nn.Sequential(Block(CHANNELS+BITS+3,CHANNELS),nn.Conv2d(CHANNELS,3,1))
  self.decoder=nn.Sequential(*[Block(3 if i==0 else CHANNELS,CHANNELS) for i in range(7)],nn.AdaptiveAvgPool2d(4 if spatial else 1),nn.Flatten(),nn.Linear(CHANNELS*(16 if spatial else 1),128),nn.ReLU(),nn.Linear(128,BITS))
 def encode(self,x,bits):
  message=F.interpolate(self.message_map(bits*2-1).reshape(-1,BITS,16,16),size=(SIZE,SIZE),mode='nearest') if self.spatial else (bits*2-1)[:,:,None,None].expand(-1,-1,SIZE,SIZE)
  residual=AMPLITUDE*torch.tanh(self.fuse(torch.cat([self.features(x),message,x],1)))
  return (x+residual.float()).clamp(0,1)
 def decode(self,x):return self.decoder(x)

def autocast():return torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=AMP)

def errors(logits,bits):return (logits>=0)!=(bits>.5)

def psnr(mse):return -10*math.log10(max(float(mse),1e-12))

def batch_images(arrays,split,ids):return arrays[split][ids].to(DEVICE,dtype=torch.float32,non_blocking=CUDA,memory_format=torch.channels_last)/255.
def new_model():return CNN(spatial=True).to(device=DEVICE,memory_format=torch.channels_last)


In [ ]:
def rgb8(x):return (x.clamp(0,1)*255).round()/255.
def quota_regions(n,epoch,p):
    rank=np.random.default_rng(SEED+303+epoch).permutation(n)
    q=round(p*n*(epoch+1))-round(p*n*epoch)
    regions=np.ones(n,dtype=np.int64);regions[rank[:q]]=0
    return regions,q

def attack(family,marked,x,noise,mask,mix):
    if family=='clean':return marked
    if family=='global':return (marked+SIGMA*noise).clamp(0,1)
    gaussian=(marked+SIGMA*noise*mask).clamp(0,1)
    erase=torch.where(mask.bool(),x,marked)
    if family=='gaussian':return gaussian
    if family=='erase':return erase
    return torch.where(mix[:,None,None,None],erase,gaussian)

@torch.inference_mode()
def evaluate(model,arrays,banks,words,n=None,alpha=1.,details=False):
    model.eval();n=n or len(words);collected=defaultdict(list);mses=[];ssims=[]
    for lo in range(0,n,EVAL_BATCH):
        ids=np.arange(lo,min(lo+EVAL_BATCH,n));x=image_batch(arrays,'val',ids);b=torch.from_numpy(words[ids]).to(DEVICE,dtype=torch.float32)
        with autocast():marked=rgb8(x+alpha*(model.encode(x,b)-x))
        # Per-image seeds independent of eval batch, reused across all arms.
        noise=torch.from_numpy(np.stack([np.random.default_rng(VAL_SEED+int(i)*191).standard_normal((3,SIZE,SIZE),dtype=np.float32) for i in ids])).to(DEVICE)
        mm=[mask_batch(banks,'val',ids,2,np.full(len(ids),r)) for r in (0,1)]
        conditions={'clean':marked,'global':rgb8((marked+SIGMA*noise).clamp(0,1))}
        for r,name in enumerate(('object','background')):
            conditions[name+'_gaussian']=rgb8((marked+SIGMA*noise*mm[r]).clamp(0,1))
            conditions[name+'_erase']=torch.where(mm[r].bool(),x,marked)
        for name,img in conditions.items():
            with autocast():logits=model.decode(img).float()
            assert torch.isfinite(logits).all()
            collected[name].append(logits.cpu().numpy())
        mses.extend(((marked-x)**2).mean((1,2,3)).cpu().numpy().tolist())
        if details:
            xx=x.cpu().permute(0,2,3,1).numpy();yy=marked.cpu().permute(0,2,3,1).numpy()
            ssims.extend(float(structural_similarity(a,b,data_range=1.,channel_axis=-1,win_size=11)) for a,b in zip(xx,yy))
    logits={k:np.concatenate(v) for k,v in collected.items()};metrics={}
    for name,z in logits.items():
        errs=((z>=0)!=words[:n]).sum(1)
        metrics[name]=dict(ber=float(errs.mean()/BITS),exact=float((errs==0).mean()),bce=float((np.logaddexp(0,z)-words[:n]*z).mean()))
    metrics['psnr']=psnr(np.mean(mses));metrics['ssim']=float(np.mean(ssims)) if ssims else None
    metrics['worst_ber']=max(metrics[k]['ber'] for k in ('object_gaussian','background_gaussian','object_erase','background_erase'))
    metrics['mean_bce']=float(np.mean([metrics[k]['bce'] for k in ('object_gaussian','background_gaussian','object_erase','background_erase')]))
    return metrics,logits,np.array(mses),np.array(ssims)

def fit(arm,initial,arrays,banks,words,epochs,parent=False):
    folder=RUN/arm;folder.mkdir(exist_ok=True)
    cfg=dict(family='global',p=.5,random_mask=False) if parent else ARM_CONFIG[arm]
    identity=dict(source=SOURCE_SHA,data=sha(DATA/'identity.json'),config=CONFIG,arm=arm,initial=digest(initial),parent=parent)
    if (folder/'completion.json').exists():
        done=json.loads((folder/'completion.json').read_text());assert done['identity']==identity and sha(folder/'final.pt')==done['sha256']
        return torch.load(folder/'final.pt',map_location='cpu',weights_only=False)
    model=new_model();model.load_state_dict(initial);opt=torch.optim.Adam(model.parameters(),lr=LR)
    scaler=torch.amp.GradScaler('cuda',enabled=AMP,init_scale=128,growth_interval=100000)
    history=[];bestscore=(float('inf'),float('inf'));beste=None
    if (folder/'last.pt').exists():
        snap=torch.load(folder/'last.pt',map_location='cpu',weights_only=False);assert snap['identity']==identity
        model.load_state_dict(snap['model']);opt.load_state_dict(snap['optimizer']);scaler.load_state_dict(snap['scaler']);history=snap['history'];bestscore=tuple(snap['bestscore']);beste=snap['best_epoch']
    n=32 if PROFILE=='smoke' else len(arrays['train']);nv=8 if PROFILE=='smoke' else len(words)
    for epoch in range(len(history),epochs):
        deadline();start=time.perf_counter();model.train()
        g=torch.Generator().manual_seed(SEED+epoch*100003)
        order=torch.randperm(n,generator=g).numpy();payload=torch.randint(0,2,(n,BITS),generator=g).float()
        noiseg=torch.Generator(device=DEVICE).manual_seed(SEED+epoch*100003+707)
        regions,q=quota_regions(n,epoch,cfg['p'])
        mix=np.zeros(n,dtype=bool);mix[np.random.default_rng(SEED+epoch*1117+99).permutation(n)[:n//2]]=True
        total=0.;steps=0;grad=[]
        for lo in range(0,n,BATCH):
            deadline();ids=order[lo:lo+BATCH];x=image_batch(arrays,'train',ids);bits=payload[ids].to(DEVICE)
            noise=torch.randn(x.shape,device=DEVICE,generator=noiseg)
            reg=np.full(len(ids),2) if cfg['random_mask'] else regions[ids]
            mask=mask_batch(banks,'train',ids,epoch%EPOCHS,reg)
            opt.zero_grad(set_to_none=True)
            with autocast():
                marked=model.encode(x,bits);dirty=attack(cfg['family'],marked,x,noise,mask,torch.from_numpy(mix[ids]).to(DEVICE))
                clean=model.decode(marked);changed=model.decode(dirty)
                loss=.5*F.binary_cross_entropy_with_logits(clean.float(),bits)+.5*F.binary_cross_entropy_with_logits(changed.float(),bits)+F.mse_loss(marked,x)
            assert torch.isfinite(loss)
            scaler.scale(loss).backward();scaler.unscale_(opt);norm=torch.nn.utils.clip_grad_norm_(model.parameters(),CLIP)
            assert torch.isfinite(norm) and norm>0
            before=scaler.get_scale();scaler.step(opt);scaler.update();assert scaler.get_scale()>=before,'AMP skipped update'
            total+=float(loss.detach())*len(ids);steps+=1;grad.append(float(norm))
        sync();train_seconds=time.perf_counter()-start
        metrics,_,_,_=evaluate(model,arrays,banks,words,nv)
        score=(metrics['global']['ber'],metrics['global']['bce']) if parent else (metrics['worst_ber'],metrics['mean_bce'])
        record=dict(epoch=epoch+1,train_loss=total/n,seconds=train_seconds,steps=steps,object_quota=q,order_sha=hashlib.sha256(order.tobytes()).hexdigest(),payload_sha=hashlib.sha256(payload.numpy().tobytes()).hexdigest(),grad_max=max(grad),validation=metrics)
        history.append(record)
        if metrics['psnr']>=30 and score<bestscore:
            bestscore=score;beste=epoch+1;save(folder/'best.pt',dict(model=state(model),epoch=beste,metrics=metrics,identity=identity))
        save(folder/'last.pt',dict(model=state(model),optimizer=opt.state_dict(),scaler=scaler.state_dict(),history=history,bestscore=bestscore,best_epoch=beste,identity=identity,rng_cpu=torch.get_rng_state(),rng_cuda=torch.cuda.get_rng_state_all() if CUDA else None))
        write(folder/'history.json',history);stamp('training',arm=arm,epoch=epoch+1,epochs=epochs,metrics=metrics)
        print(arm,epoch+1,'seconds',round(train_seconds,1),'clean',round(metrics['clean']['ber'],4),'worst',round(metrics['worst_ber'],4),'PSNR',round(metrics['psnr'],2),flush=True)
        if parent and epoch+1>=10 and metrics['clean']['ber']<=.03 and metrics['global']['ber']<=.04:break
    assert beste is not None,'No checkpoint passed image quality gate'
    best=torch.load(folder/'best.pt',map_location='cpu',weights_only=False);save(folder/'final.pt',best)
    write(folder/'completion.json',dict(epochs=len(history),selected_epoch=beste,sha256=sha(folder/'final.pt'),identity=identity,metrics=best['metrics']))
    del model,opt,scaler;gc.collect()
    if CUDA:torch.cuda.empty_cache()
    return best

def benchmark(arrays,banks,words):
    records=[]
    for bs in (16,32,64):
        model=new_model().train();opt=torch.optim.Adam(model.parameters(),lr=LR);scaler=torch.amp.GradScaler('cuda',init_scale=128)
        x=image_batch(arrays,'train',np.arange(bs));bits=torch.randint(0,2,(bs,BITS),device=DEVICE).float();mask=mask_batch(banks,'train',np.arange(bs),0,np.zeros(bs,dtype=int));noise=torch.randn_like(x)
        def step():
            opt.zero_grad(set_to_none=True)
            with autocast():
                marked=model.encode(x,bits);dirty=(marked+SIGMA*noise*mask).clamp(0,1)
                loss=.5*F.binary_cross_entropy_with_logits(model.decode(marked).float(),bits)+.5*F.binary_cross_entropy_with_logits(model.decode(dirty).float(),bits)+F.mse_loss(marked,x)
            scaler.scale(loss).backward();scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(model.parameters(),CLIP);scaler.step(opt);scaler.update()
        try:
            torch.cuda.reset_peak_memory_stats()
            for _ in range(3):step()
            sync();t=time.perf_counter()
            for _ in range(15):step()
            sync();elapsed=time.perf_counter()-t
            records.append(dict(batch=bs,images_per_second=bs*15/elapsed,peak_reserved=torch.cuda.max_memory_reserved()))
        except torch.cuda.OutOfMemoryError:records.append(dict(batch=bs,oom=True))
        del model,opt,scaler,x,bits,mask,noise;gc.collect();torch.cuda.empty_cache()
    model=new_model();sync();t=time.perf_counter();evaluate(model,arrays,banks,words);sync();val_seconds=time.perf_counter()-t
    for r in records:
        if 'images_per_second' in r:r['projected_20arms_20epochs_seconds']=20*EPOCHS*(5000/r['images_per_second']+val_seconds)
    write(RUN/'benchmark.json',dict(candidates=records,validation500_seconds=val_seconds,gpu=torch.cuda.get_device_name(),remaining_training_seconds=TRAIN_DEADLINE-time.time()))
    print(json.dumps(dict(candidates=records,validation500_seconds=val_seconds),indent=2),flush=True)

def freeze(arrays,banks,words,manifest):
    raw={};cache={}
    for arm in ARMS:
        snap=torch.load(RUN/arm/'final.pt',map_location='cpu',weights_only=False);model=new_model();model.load_state_dict(snap['model']);model.eval()
        chunks=[]
        with torch.inference_mode():
            for lo in range(0,len(words),EVAL_BATCH):
                ids=np.arange(lo,min(lo+EVAL_BATCH,len(words)));x=image_batch(arrays,'val',ids);b=torch.from_numpy(words[ids]).to(DEVICE,dtype=torch.float32)
                with autocast():y=model.encode(x,b)
                chunks.append(y.float().cpu().permute(0,2,3,1).numpy())
        cache[arm]=np.concatenate(chunks);x=arrays['val'].permute(0,2,3,1).numpy().astype(np.float32)/255.
        raw[arm]=psnr(np.mean((np.round(cache[arm]*255)/255-x)**2));del model
    target=max(raw.values());handoff=dict(source_sha=SOURCE_SHA,data_sha=sha(DATA/'identity.json'),seed=SEED,config=CONFIG,validation_target_psnr=target,arms={})
    for arm in ARMS:
        lo,hi=0.,1.
        for _ in range(24):
            a=(lo+hi)/2;quality=psnr(np.mean((np.round((x+a*(cache[arm]-x))*255)/255-x)**2))
            if quality>target:lo=a
            else:hi=a
        alpha=(lo+hi)/2 if raw[arm]<target else 1.
        snap=torch.load(RUN/arm/'final.pt',map_location='cpu',weights_only=False);model=new_model();model.load_state_dict(snap['model'])
        metrics,z,mse,ssim=evaluate(model,arrays,banks,words,alpha=alpha,details=True)
        np.savez_compressed(RUN/arm/'validation.npz',bits=words,mse=mse,ssim=ssim,ids=manifest['splits']['val']['ids'],**z)
        handoff['arms'][arm]=dict(path=arm+'/final.pt',sha256=sha(RUN/arm/'final.pt'),epoch=snap['epoch'],alpha=alpha,raw_psnr=raw[arm],validation=metrics)
        write(RUN/arm/'calibration.json',handoff['arms'][arm]);del model
    qualities=[v['validation'] for v in handoff['arms'].values()]
    handoff['quality_gate']=dict(psnr_spread=float(np.ptp([q['psnr'] for q in qualities])),ssim_spread=float(np.ptp([q['ssim'] for q in qualities])),psnr_min=min(q['psnr'] for q in qualities))
    handoff['frozen_utc']=dt.datetime.now(dt.timezone.utc).isoformat();write(RUN/'handoff.json',handoff)
    fig,axes=plt.subplots(1,2,figsize=(14,6))
    for arm in ARMS:
        hist=json.loads((RUN/arm/'history.json').read_text());axes[0].plot([h['epoch'] for h in hist],[h['train_loss'] for h in hist],label=arm);axes[1].plot([h['epoch'] for h in hist],[100*h['validation']['worst_ber'] for h in hist])
    axes[0].set_title('Training loss');axes[1].set_title('Validation worst-region BER (%)');axes[0].legend(fontsize=6,ncol=2);fig.tight_layout();fig.savefig(RUN/'learning-curves.png',dpi=160);plt.close(fig)
    stamp('frozen',handoff_sha=sha(RUN/'handoff.json'),models=len(ARMS));print('FROZEN',sha(RUN/'handoff.json'),flush=True)


In [ ]:
if PROFILE=='prepare':prepare();stamp('prepared')
else:
    arrays,banks,words,manifest=load_data()
    if PROFILE=='benchmark':benchmark(arrays,banks,words)
    elif PROFILE=='smoke':
        model=new_model();initial=state(model)
        # Attack invariants include signal removal, mask locality, exact quotas and clean identity.
        x=image_batch(arrays,'train',np.arange(4));bits=torch.randint(0,2,(4,BITS),device=DEVICE).float();y=model.encode(x,bits);mask=mask_batch(banks,'train',np.arange(4),0,np.zeros(4,dtype=int))
        erased=attack('erase',y,x,torch.zeros_like(x),mask,torch.zeros(4,device=DEVICE,dtype=torch.bool))
        assert torch.equal(erased[mask.expand_as(x).bool()],x[mask.expand_as(x).bool()]);assert torch.equal(erased[~mask.expand_as(x).bool()],y[~mask.expand_as(x).bool()])
        for p in (0.,.1,.5,.9,1.):
            rr,q=quota_regions(5000,0,p);assert (rr==0).sum()==q==round(p*5000)
        for arm in ARMS:
            snap=fit(arm,initial,arrays,banks,words,1);assert digest(snap['model'])!=digest(initial)
            # Reenter completed epoch boundary and verify exact state identity.
            snap2=fit(arm,initial,arrays,banks,words,1);assert digest(snap['model'])==digest(snap2['model'])
        stamp('smoke_passed',arms=len(ARMS),scientific_result=False)
    else:
        if PROFILE=='train':
            torch.manual_seed(SEED);initial=state(new_model())
            parent=fit('parent',initial,arrays,banks,words,PARENT_EPOCHS,parent=True)
            assert parent['metrics']['clean']['ber']<=.1 and parent['metrics']['global']['ber']<=.1,'Parent learning gate failed'
            for arm in ARMS:fit(arm,parent['model'],arrays,banks,words,EPOCHS)
            # Verify stochastic pairing across every complete training epoch.
            histories=[json.loads((RUN/a/'history.json').read_text()) for a in ARMS]
            assert all(len(h)==EPOCHS for h in histories)
            for hh in zip(*histories):
                assert len({h['order_sha'] for h in hh})==len({h['payload_sha'] for h in hh})==1
        freeze(arrays,banks,words,manifest)
